# Homework 4 — Evaluating Search

## Setup

Load the API key from `.env`, create the Anthropic client, and pin the model.

In [1]:
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

llm_client = Anthropic()
MODEL = "claude-haiku-4-5"

## Lesson pages

Pull the lesson pages from the pinned commit `8c1834d`, so the dataset is
identical for everyone.

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

# Each file has a parse() method returning {"filename": ..., "content": ...}
documents = [file.parse() for file in reader.read()]

print("Lesson pages:", len(documents))

Lesson pages: 72


## Q1. Average input tokens

Ground truth is built by asking an LLM to write 5 questions answered by each
lesson page. Starting small: generate questions for the first 3 pages and
average the input tokens reported per call.

In [3]:
import json

from pydantic import BaseModel

from evaluation_utils import llm_structured


class Questions(BaseModel):
    questions: list[str]


data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()


# Ask the LLM for 5 questions on one page; return records plus token usage.
def generate_questions(page):
    user_prompt = json.dumps({
        "filename": page["filename"],
        "content": page["content"],
    })

    result, usage = llm_structured(
        client=llm_client,
        instructions=data_gen_instructions,
        user_prompt=user_prompt,
        output_type=Questions,
        model=MODEL,
    )

    records = [
        {"filename": page["filename"], "question": question}
        for question in result.questions
    ]

    return records, usage

In [4]:
from tqdm.auto import tqdm

first_three_pages = documents[:3]

question_records = []
usages = []

for page in tqdm(first_three_pages):
    records, usage = generate_questions(page)
    question_records.extend(records)
    usages.append(usage)

avg_input_tokens = sum(usage.input_tokens for usage in usages) / len(usages)
print("Q1 — average input tokens across 3 calls:", round(avg_input_tokens, 2))

  0%|          | 0/3 [00:00<?, ?it/s]

Q1 — average input tokens across 3 calls: 1729.33


**Answer Q1: 1400** — the run averaged ~1729 input tokens (Claude Haiku 4.5),
closest to the 1400 option. Token counts vary by run and model but stay in the
same order of magnitude.

## Ground truth

The full ground truth (360 questions across all 72 pages) was pre-generated with
the same approach. Load it as a list of `{question, filename}` records.

In [5]:
import pandas as pd

ground_truth_df = pd.read_csv("ground-truth.csv")
ground_truth = ground_truth_df.to_dict(orient="records")

print("Ground-truth questions:", len(ground_truth))
ground_truth[0]

Ground-truth questions: 360


{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 'filename': '01-agentic-rag/lessons/01-intro.md'}

## Building search

Search runs over the same chunks as homework 2. Below: a text index, a vector
index (MiniLM embeddings via `embedder.py`), and three search functions — text,
vector, and hybrid (RRF) — all keyed on `filename`.

In [6]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print("Chunks:", len(chunks))

Chunks: 295


In [7]:
from minsearch import Index

# Text index over chunk content, keyed on filename.
text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)


def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

In [8]:
import numpy as np
from minsearch import VectorSearch

from embedder import Embedder

embedder = Embedder()

# Embed every chunk in batches, then build the vector index keyed on filename.
chunk_texts = [chunk["content"] for chunk in chunks]
batch_size = 50

chunk_vectors = []
for i in tqdm(range(0, len(chunk_texts), batch_size)):
    batch = chunk_texts[i:i + batch_size]
    chunk_vectors.extend(embedder.encode_batch(batch))

chunk_vectors = np.array(chunk_vectors)

vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(chunk_vectors, chunks)


def vector_search(query, num_results=5):
    query_vector = embedder.encode(query)
    return vector_index.search(query_vector, num_results=num_results)

print("Vector matrix shape:", chunk_vectors.shape)

  0%|          | 0/6 [00:00<?, ?it/s]

Vector matrix shape: (295, 384)


In [9]:
# Reciprocal Rank Fusion: merge result lists, rewarding items ranked high in both.
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

## Q2. First result — text search

Take the first ground-truth question (generated from `01-intro.md`) and check
which page text search ranks first.

In [10]:
q = ground_truth[0]["question"]
print("Question:", q)

top_text = text_search(q)[0]
print("Q2 — top text-search filename:", top_text["filename"])

Question: What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
Q2 — top text-search filename: 01-agentic-rag/lessons/03-rag.md


**Answer Q2: 01-agentic-rag/lessons/03-rag.md**

## Q3. First result — vector search

Same question, vector search. It ranks the correct page (`01-intro.md`) first,
while text search did not — a preview of why measuring across the whole set
matters.

In [11]:
top_vector = vector_search(q)[0]
print("Q3 — top vector-search filename:", top_vector["filename"])

Q3 — top vector-search filename: 01-agentic-rag/lessons/01-intro.md


**Answer Q3: 01-agentic-rag/lessons/01-intro.md**

## Evaluation metrics

A result is a hit when a returned chunk's `filename` matches the question's
`filename`. `compute_relevance` turns one search into a list of 0/1; Hit Rate is
the share of questions with a hit anywhere in the results; MRR also rewards
ranking the right page near the top.

In [12]:
def compute_relevance(question_record, search_function):
    filename = question_record["filename"]
    results = search_function(query=question_record["question"])
    return [int(result["filename"] == filename) for result in results]


def hit_rate(relevance_total):
    hits = sum(1 for relevance in relevance_total if 1 in relevance)
    return hits / len(relevance_total)


def mrr(relevance_total):
    total_score = 0.0
    for relevance in relevance_total:
        for rank, is_hit in enumerate(relevance):
            if is_hit == 1:
                total_score += 1 / (rank + 1)
                break
    return total_score / len(relevance_total)


def evaluate(ground_truth, search_function):
    relevance_total = [
        compute_relevance(question_record, search_function)
        for question_record in tqdm(ground_truth)
    ]
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

## Q4. Text search — Hit Rate

Evaluate text search over all 360 ground-truth questions.

In [13]:
text_metrics = evaluate(ground_truth, text_search)
print("Q4 — text search:", text_metrics)

  0%|          | 0/360 [00:00<?, ?it/s]

Q4 — text search: {'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592594}


**Answer Q4: 0.76** — Hit Rate ≈ 0.758.

## Q5. Vector search — MRR

Evaluate vector search — the method the module left unmeasured.

In [14]:
vector_metrics = evaluate(ground_truth, vector_search)
print("Q5 — vector search:", vector_metrics)

  0%|          | 0/360 [00:00<?, ?it/s]

Q5 — vector search: {'hit_rate': 0.725, 'mrr': 0.5486111111111112}


**Answer Q5: 0.55** — MRR ≈ 0.549 (Hit Rate ≈ 0.725, a touch below text search).

## Q6. Tuning hybrid search

RRF's `k` controls how much the top ranks matter — a smaller `k` sharpens the
gap between positions. Sweep `k` over 1, 50, 100, 200 and compare MRR. On a tie,
the smallest `k` wins.

In [15]:
ks = [1, 50, 100, 200]
hybrid_results = []

for k in ks:
    metrics = evaluate(ground_truth, lambda query, k=k: hybrid_search(query, k))
    hybrid_results.append({"k": k, **metrics})

hybrid_df = pd.DataFrame(hybrid_results)
print(hybrid_df.to_string(index=False))

# max() keeps the first maximal row; ks ascending -> smallest k wins any tie.
best_k = max(hybrid_results, key=lambda row: row["mrr"])["k"]
print("Q6 — best k by MRR:", best_k)

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  0%|          | 0/360 [00:00<?, ?it/s]

  k  hit_rate      mrr
  1  0.838889 0.648194
 50  0.836111 0.637917
100  0.836111 0.637917
200  0.836111 0.637917
Q6 — best k by MRR: 1


**Answer Q6: 1** — k=1 gives the best MRR (≈0.648); k=50/100/200 tie lower
(≈0.638).

## Summary

| Method | Hit Rate | MRR |
|---|---|---|
| Text | 0.758 | 0.594 |
| Vector | 0.725 | 0.549 |
| Hybrid (k=1) | 0.839 | 0.648 |